# Phase 8 — GIS / tables export

1. Write layer GeoJSON files to `outputs/`
2. Keep Plotly HTML maps under `maps/` (stamp limitations banner)
3. Write CSV/JSON summaries to `results/tables/`

**All logic runs in this notebook** (no external script imports).

**Requires:** notebooks `06`–`13` completed.


In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import polars as pl

ROOT = Path('../..').resolve()
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS_DIR = ROOT / 'outputs'
OUT_GEOJSON = OUTPUTS_DIR / 'ny_composite_ranked.geojson'
MAP_DIR = ROOT / 'maps'
RESULTS_TABLES = ROOT / 'results' / 'tables'
JOIN = ['latitude', 'longitude']
NLCD_COLS = [
    'frac_class_11', 'frac_class_21', 'frac_class_22', 'frac_class_23', 'frac_class_24', 'frac_developed',
]

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

gateway = pl.read_parquet(PROCESSED / 'cells_passing_gateway.parquet').select(JOIN + NLCD_COLS)


def write_geojson(df: pl.DataFrame, path: Path) -> None:
    pdf = df.to_pandas()
    gdf = gpd.GeoDataFrame(
        pdf,
        geometry=gpd.points_from_xy(pdf['longitude'], pdf['latitude']),
        crs='EPSG:4326',
    )
    gdf.to_file(path, driver='GeoJSON')
    print(f'GeoJSON {len(df):,} features -> {path}')


# --- GIS layer exports ---
solar = pl.read_parquet(PROCESSED / 'solar_scores_by_cell.parquet').join(gateway, on=JOIN, how='left')
solar = solar.select(JOIN + NLCD_COLS + [
    'resource_observed', 'resource_observed_pct', 'resource_screening_score',
    'resource_land_adj_oof', 'resource_opportunity_oof',
]).rename({
    'resource_observed': 'solar_resource_w_m2',
    'resource_observed_pct': 'solar_resource_pct',
    'resource_screening_score': 'solar_screening_pct',
})
write_geojson(solar, OUTPUTS_DIR / 'ny_solar_screening.geojson')

wind = pl.read_parquet(PROCESSED / 'wind_scores_by_cell.parquet').join(gateway, on=JOIN, how='left')
wind = wind.select(JOIN + NLCD_COLS + [
    'resource_observed', 'resource_observed_pct', 'resource_screening_score',
    'resource_land_adj_oof', 'resource_opportunity_oof',
]).rename({
    'resource_observed': 'wind_power_index',
    'resource_observed_pct': 'wind_resource_pct',
    'resource_screening_score': 'wind_screening_pct',
})
write_geojson(wind, OUTPUTS_DIR / 'ny_wind_screening.geojson')

hybrid = pl.read_parquet(PROCESSED / 'hybrid_suitability.parquet').join(gateway, on=JOIN, how='left')
hybrid = hybrid.select(JOIN + NLCD_COLS + [
    'hybrid_seasonal_score', 'jackpot_zone', 'hybrid_screening_score',
    'summer_ghi', 'winter_wind', 'solar_cf_p50', 'wind_cf_p50',
])
write_geojson(hybrid, OUTPUTS_DIR / 'ny_hybrid_suitability.geojson')

MAP_FILES = [
    'developability_ny.html',
    'solar_resource_ny.html',
    'wind_resource_ny.html',
    'hybrid_bivariate_resource_ny.html',
    'composite_score_ny.html',
    'hybrid_seasonal_suitability_ny.html',
    'risk_spread_ny.html',
    'solar_yield_p50_ny.html',
    'wind_yield_p50_ny.html',
]

if not OUT_GEOJSON.exists():
    raise FileNotFoundError(f'Missing {OUT_GEOJSON} — run 13_composite_score.ipynb first')

maps_present = []
for name in MAP_FILES:
    src = MAP_DIR / name
    if src.exists():
        maps_present.append(name)
    else:
        print('SKIP (missing map):', src)

fc = json.loads(OUT_GEOJSON.read_text())
props = [f['properties'] for f in fc['features']]
props.sort(key=lambda p: p['composite_score'], reverse=True)
top = props[0]
summary = {
    'n_cells': len(props),
    'n_jackpot': sum(1 for p in props if p.get('jackpot_zone')),
    'top_latitude': top['latitude'],
    'top_longitude': top['longitude'],
    'top_composite_score': top['composite_score'],
    'top_cluster_label': top.get('cluster_label', ''),
    'maps_present': maps_present,
}
summary_path = RESULTS_TABLES / 'export_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Summary ->', summary_path)

top50 = pl.DataFrame(props[:50])
csv_path = RESULTS_TABLES / 'top50_composite_ranked.csv'
top50.write_csv(csv_path)
print('CSV ->', csv_path)

LIMITATIONS_BANNER = (
    '<div id="pipeline-limitations-banner" style="position:fixed;bottom:0;left:0;right:0;'
    'background:rgba(20,16,8,0.92);color:#f5f0e6;font:12px/1.4 system-ui,sans-serif;'
    'padding:8px 14px;z-index:9999;border-top:1px solid #b45309;'>'
    '<strong>Screening only.</strong> 10 km grid · 2018–2025 weather · NLCD 2024 · not permitting, '
    'hub-height wind, or interconnection. Yield maps experimental.'
    '</div>'
)


def stamp_map_html(path: Path) -> None:
    text = path.read_text(encoding='utf-8')
    if 'pipeline-limitations-banner' in text:
        return
    path.write_text(text.replace('</body>', LIMITATIONS_BANNER + '\n</body>'), encoding='utf-8')
    print('Stamped limitations ->', path.name)


for name in MAP_FILES:
    p = MAP_DIR / name
    if p.exists():
        stamp_map_html(p)

print(json.dumps(summary, indent=2))


In [ ]:
# GIS / tables export is in the cell above.
